In [ ]:
! pip install -q ultralytics gradio | echo "Instalado"


In [ ]:
import os
from typing import Tuple
import numpy as np
from PIL import Image
import gradio as gr
from ultralytics import YOLO

In [ ]:


# -----------------------------
# Carga del modelo (una sola vez)
# -----------------------------
# Puedes cambiar a "yolov8s.pt", "yolov8m.pt", etc. según precisión/velocidad.
MODEL_NAME = os.getenv("YOLO_MODEL", "yolov8n.pt")
model = YOLO(MODEL_NAME)  # descarga automática si no está presente

def detectar(imagen: np.ndarray, conf: float, iou: float, max_det: int, imgsz: int) -> np.ndarray:
    """
    Ejecuta la predicción sobre una imagen (numpy array RGB) y devuelve la imagen anotada.
    """
    # Ultralytics acepta numpy/PIL directamente
    results = model.predict(
        source=imagen,
        conf=conf,
        iou=iou,
        max_det=max_det,
        imgsz=int(imgsz),
        verbose=False
    )

    # results[0].plot() devuelve un array BGR; lo convertimos a RGB para Gradio
    bgr_annotated = results[0].plot()
    rgb_annotated = bgr_annotated[:, :, ::-1]  # BGR -> RGB
    return rgb_annotated

# -----------------------------
# Interfaz de Gradio
# -----------------------------
with gr.Blocks(title="Detección con YOLO (Ultralytics)") as demo:
    gr.Markdown(
        """
        # Detección de objetos con YOLO
        Sube una imagen y el modelo dibujará cajas y etiquetas sobre los objetos detectados.
        Puedes ajustar el umbral de confianza, IoU, tamaño de imagen y máximo de detecciones.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="numpy", label="Imagen de entrada", sources=["upload", "clipboard"])
            conf_slider = gr.Slider(0.1, 0.95, value=0.5, step=0.05, label="Confianza mínima (conf)")
            iou_slider  = gr.Slider(0.1, 0.95, value=0.45, step=0.05, label="IoU (NMS)")
            imgsz_slider = gr.Slider(320, 1280, value=640, step=32, label="Tamaño de imagen (imgsz)")
            maxdet_slider = gr.Slider(10, 300, value=100, step=10, label="Máximo de detecciones (max_det)")
            btn = gr.Button("Detectar")
        with gr.Column(scale=1):
            output_image = gr.Image(type="numpy", label="Imagen anotada")

    # Acciones
    btn.click(
        fn=detectar,
        inputs=[input_image, conf_slider, iou_slider, maxdet_slider, imgsz_slider],
        outputs=output_image
    )

    # También ejecutar automáticamente al cambiar la imagen
    input_image.change(
        fn=detectar,
        inputs=[input_image, conf_slider, iou_slider, maxdet_slider, imgsz_slider],
        outputs=output_image
    )


demo.launch()